In [1]:
import json
import logging
import os
from pathlib import Path

import boto3
import ipynbname
import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
p = ipynbname.path().parent.parent

format = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
filename_path = Path(p / 'logs' / 'transform_data.log')
logging.basicConfig(format=format, filename=filename_path, encoding='utf-8', level=logging.INFO)
logger = logging.getLogger(__name__)

load_dotenv(dotenv_path=p / '.env')

def extract_data():
    if not os.getenv('EXTRACT_DATETIME'):
        logger.error("EXTRACT_DATETIME is not set in the environment variables.")
        raise TypeError("Expected string for EXTRACT_DATETIME, received None (EXTRACT_DATETIME is not set in the environment variables).")
    
    if not os.getenv('S3_BUCKET_NAME'):
            logger.error("S3_BUCKET_NAME is not set in the environment variables.")
            raise TypeError("Expected string for S3_BUCKET_NAME, received None (S3_BUCKET_NAME is not set in the environment variables).")
    
    if not os.getenv('AWS_ACCESS_KEY_ID'):
        logger.error("AWS_ACCESS_KEY_ID is not set in the environment variables.")
        raise TypeError("Expected string for AWS_ACCESS_KEY_ID, received None (AWS_ACCESS_KEY_ID is not set in the environment variables).")

    if not os.getenv('AWS_SECRET_ACCESS_KEY'):
        logger.error("AWS_SECRET_ACCESS_KEY is not set in the environment variables.")
        raise TypeError("Expected string for AWS_SECRET_ACCESS_KEY, received None (AWS_SECRET_ACCESS_KEY is not set in the environment variables).")
    

    try:
        s3_client = boto3.client(
            's3', 
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'), 
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
        )
        response = s3_client.get_object(
            Bucket=os.getenv('S3_BUCKET_NAME'), 
            Key=f"bronze/mercadolivre/celulares-e-telefones/{os.getenv('EXTRACT_DATETIME')}/bronze_data.json"
        )
        data = json.loads(response['Body'].read().decode('utf-8'))
        logger.info("bronze_data.json obtained from Amazon S3")
        return data
    except Exception as e:
        logger.error(f"Error occurred while extracting data from Amazon S3: {e}")
        raise

def create_dataframe() -> pd.DataFrame:
    data = extract_data()
    df = pd.json_normalize(data)
    return df

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    df_data_items = df.explode('data.items', ignore_index=True)
    df_items = pd.json_normalize(df_data_items['data.items'])
    df = pd.concat(
        [df_data_items, df_items],
        axis=1
    )
    return df

df = create_dataframe()
df = normalize_df(df)
df.head()


,requestId,executionId,data.source,data.type,data.url,data.requestUrl,data.extractedAt,data.query,data.totalResults,data.primaryResults,data.page,data.resultsPerPage,data.offset,data.nextPage,data.nextPageUrl,data.items,url,imageUrl,sku,categoryId,domainId,name,condition,currency,currencyRaw,price,highlight,isBestSeller,isPowerSeller,powerSellerStatusTitle,ean,sellerId,sellerName,sellerCity,sellerState,sellerCountry,aggregateRating.rating,aggregateRating.reviewCount,aggregateRating
0,4b8d3c97-c040-4334-9547-e9193f2c9c3e,9af2c5ed-21c0-4c56-915c-dacc20bb7561,mercadolivre.com.br,plp,https://lista.mercadolivre.com.br/iphone-15-pro-max,https://lista.mercadolivre.com.br/iphone-15-pro-max,2026-10-28T21:01:47.408Z,iphone 15 pro max,179,179,1,48,0,2.0,https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/iphone/iphone-15-pro-max/iphone-15-pro-max_Desde_49_NoIndex_True,"{'url': 'https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-azul-excelente-recondicionado/p/MLB2000048511', 'imageUrl': 'https://http2.mlstatic.com/D_Q_NP_2X_747522-MLA95493130164_102025-E.webp', 'sku': 'MLB7877243580', 'categoryId': 'MLB1055', 'domainId': 'MLB-CELLPHONES', 'name': 'Apple iPhone 15 Pro Max (256 GB) - Titânio Azul - Excelente (Recondicionado)', 'condition': 'Recondicionado', 'currency': 'BRL', 'currencyRaw': 'R$', 'price': 4480, 'highlight': 'ÚLTIMAS 5', 'isBestSeller': False, 'isPowerSeller': False, 'powerSellerStatusTitle': None, 'aggregateRating': {'rating': 4.6, 'reviewCount': None}, 'ean': None, 'sellerId': None, 'sellerName': None, 'sellerCity': None, 'sellerState': None, 'sellerCountry': None}",https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-azul-excelente-recondicionado/p/MLB2000048511,https://http2.mlstatic.com/D_Q_NP_2X_747522-MLA95493130164_102025-E.webp,MLB7877243580,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (256 GB) - Titânio Azul - Excelente (Recondicionado),Recondicionado,BRL,R$,4480,ÚLTIMAS 5,False,False,None,None,None,NaN,None,None,None,4.6,NaN,NaN
1,4b8d3c97-c040-4334-9547-e9193f2c9c3e,9af2c5ed-21c0-4c56-915c-dacc20bb7561,mercadolivre.com.br,plp,https://lista.mercadolivre.com.br/iphone-15-pro-max,https://lista.mercadolivre.com.br/iphone-15-pro-max,2026-10-28T21:01:47.408Z,iphone 15 pro max,179,179,1,48,0,2.0,https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/iphone/iphone-15-pro-max/iphone-15-pro-max_Desde_49_NoIndex_True,"{'url': 'https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-natural-novo-com-caixa-aberta/p/MLB2016209990', 'imageUrl': 'https://http2.mlstatic.com/D_Q_NP_2X_729795-MLA95495293322_102025-E.webp', 'sku': 'MLB5261987603', 'categoryId': 'MLB1055', 'domainId': 'MLB-CELLPHONES', 'name': 'Apple iPhone 15 Pro Max (256 GB) - Titânio Natural (Novo com caixa aberta)', 'condition': 'Caixa aberta', 'currency': 'BRL', 'currencyRaw': 'R$', 'price': 5450, 'highlight': 'ÚLTIMA!', 'isBestSeller': False, 'isPowerSeller': False, 'powerSellerStatusTitle': None, 'aggregateRating': {'rating': 4.1, 'reviewCount': None}, 'ean': None, 'sellerId': None, 'sellerName': None, 'sellerCity': None, 'sellerState': None, 'sellerCountry': None}",https://www.mercadolivre.com.br/apple-iphone-15-pro-max-256-gb-titanio-natural-novo-com-caixa-aberta/p/MLB2016209990,https://http2.mlstatic.com/D_Q_NP_2X_729795-MLA95495293322_102025-E.webp,MLB5261987603,MLB1055,MLB-CELLPHONES,Apple iPhone 15 Pro Max (256 GB) - Titânio Natural (Novo com caixa aberta),Caixa aberta,BRL,R$,5450,ÚLTIMA!,False,False,None,None,None,NaN,None,None,None,4.1,NaN,NaN
2,4b8d3c97-c040-4334-9547-e9193f2c9c3e,9af2c5ed-21c0-4c56-915c-dacc20bb7561,mercadolivre.com.br,plp,https://lista.mercadolivre.com.br/iphone-15-pro-max,https://lista.mercadolivre.com.br/iphone-15-pro-max,2026-10-28T21:01:47.408Z,iphone 15 pro max,179,179,1,48,0,2.0,https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/iphone/iphone-15-pro-max/iphone-15-pro-max_Desde_49_NoIndex_True,"{'url': '